<a href="https://colab.research.google.com/github/sanmquin/AI/blob/main/research/1.Channel-Descriptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Channel Descriptions via Gemini

This notebook generates a natural-language description for every channel in the business cluster by
feeding the channel's video titles to Gemini through the **Google Colab native integration** (no API
key required).  Results are cached in Drive so that re-running the notebook skips channels whose
descriptions already exist.  A combined `all_channels.json` artifact is written at the end.


## 1) Setup and Drive mount

Install runtime dependencies, mount Google Drive, and detect whether the notebook is running inside
Colab.  The `is_colab` flag gates all Drive I/O and Gemini calls; when `False` the notebook runs
in headless mode using a dummy dataset.


In [ ]:
!pip install -q pandas numpy

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import ai, drive
    drive.mount('/content/drive')
    is_colab = True
except ImportError:
    print('Not running in Colab — headless mode active.')
    is_colab = False
except Exception as exc:
    print(f'Colab initialisation error: {exc}')
    is_colab = False


## 2) Load 20D video embeddings

Load the canonical 20-dimensional video embedding export from Drive.  If the file is not present
(e.g., in a headless test environment), a small dummy dataset is created so the rest of the
notebook can still execute end-to-end.


In [ ]:
DATA_PATH = Path(
    '/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/'
    'business_cluster_video_embeddings_reduced_20d.csv'
)

if not DATA_PATH.exists():
    print(f'Warning: {DATA_PATH} not found — creating dummy dataset.')
    rng = np.random.default_rng(42)
    channel_names = [f'Channel_{c}' for c in ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon']]
    rows = []
    for i in range(200):
        ch = channel_names[i % len(channel_names)]
        rows.append({
            'video_id':    f'vid_{i}',
            'channel_id':  f'ch_{i % len(channel_names)}',
            'channel_name': ch,
            'video_title': f'Dummy video {i} about topic {i % 10}',
            'view_count':  int(rng.integers(1_000, 100_000)),
            **{f'embedding_reduced_{j+1:02d}': float(rng.standard_normal()) for j in range(20)},
        })
    df = pd.DataFrame(rows)
else:
    df = pd.read_csv(DATA_PATH)

# Resolve canonical column names (guard against schema variations)
channel_col = 'channel_name' if 'channel_name' in df.columns else 'channel_id'
title_col   = 'video_title'  if 'video_title'  in df.columns else 'title'
dim_cols    = [c for c in df.columns if c.startswith('embedding_reduced_')]

channels = df[channel_col].dropna().unique().tolist()
print(f'Loaded {len(df):,} videos across {len(channels)} channels, {len(dim_cols)} embedding dimensions.')


## 3) Gemini description generation

This cell defines the `call_gemini` helper, sets up output paths, and iterates over every channel.

For each channel:
1. Check whether the per-channel artifact already exists in Drive → **skip** if it does.
2. Collect up to 50 video titles from that channel.
3. Build a structured prompt asking Gemini to synthesise the titles into a 2–4 paragraph channel description.
4. Write the JSON artifact to `by_channel/{channel_slug}.json`.

`call_gemini` wraps the Google Colab native `ai.generate_text()` call, which authenticates through
the user's Google account without an explicit API key.  A 2-second sleep between calls respects
rate limits.  In headless mode the function returns a placeholder string.


In [ ]:
def call_gemini(prompt: str) -> str:
    """Call Gemini via the Colab native integration; return a dummy string in headless mode."""
    if is_colab:
        try:
            time.sleep(2)          # respect rate limits
            return ai.generate_text(prompt)
        except Exception as exc:
            print(f'  Gemini error: {exc}')
            return 'Error generating description.'
    return 'Headless mode — no Gemini call made.'


# ---- Artifact paths --------------------------------------------------------
if is_colab:
    RESEARCH_ROOT  = Path('/content/drive/MyDrive/Research')
else:
    RESEARCH_ROOT  = Path('./research_output')
    print('Headless mode: writing to ./research_output')

BY_CHANNEL_DIR = RESEARCH_ROOT / 'channel_descriptions' / 'by_channel'
BY_CHANNEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Per-channel output directory: {BY_CHANNEL_DIR}')

# ---- Main generation loop --------------------------------------------------
channel_descriptions: dict = {}

for channel_name in channels:
    slug          = channel_name.replace('/', '_').replace(' ', '_').replace('\\', '_')
    artifact_path = BY_CHANNEL_DIR / f'{slug}.json'

    # Skip channels whose description has already been generated
    if artifact_path.exists():
        print(f'  [SKIP] {channel_name}')
        with open(artifact_path) as fh:
            channel_descriptions[channel_name] = json.load(fh)
        continue

    print(f'  [GEN]  {channel_name}')

    # Collect up to 50 video titles for this channel
    ch_df  = df[df[channel_col] == channel_name]
    titles = ch_df[title_col].dropna().tolist()[:50]
    title_block = '\n'.join(f'- {t}' for t in titles)

    prompt = f"""You are an expert content analyst.

Below are up to 50 video titles from the YouTube channel "{channel_name}".
Based solely on these titles, write a 2–4 paragraph description of the channel covering:

1. The main topics and recurring themes.
2. The typical content format or style (e.g. interviews, analysis, tutorials, opinion).
3. The target audience and editorial tone.
4. Any distinguishing patterns or hallmarks you can observe.

Write in clear, informative prose.  Do not enumerate the titles — synthesise them.

Video titles:
{title_block}
"""

    description = call_gemini(prompt)

    artifact = {
        'channel_name':    channel_name,
        'video_count':     len(ch_df),
        'titles_sampled':  len(titles),
        'description':     description,
        'generated_at':    time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }

    with open(artifact_path, 'w') as fh:
        json.dump(artifact, fh, indent=2)

    channel_descriptions[channel_name] = artifact
    print(f'       saved → {artifact_path}')

print(f'\nTotal: {len(channel_descriptions)} channel descriptions generated/loaded.')


## 4) Save combined artifact

Write all channel descriptions into a single `all_channels.json` file under
`Research/channel_descriptions/latest/`.  This combined artifact is consumed by
all downstream notebooks so they do not need to load per-channel files individually.


In [ ]:
overall_dir  = RESEARCH_ROOT / 'channel_descriptions' / 'latest'
overall_dir.mkdir(parents=True, exist_ok=True)
overall_path = overall_dir / 'all_channels.json'

overall_artifact = {
    'schema_version': '1.0.0',
    'channel_count':  len(channel_descriptions),
    'channels':       channel_descriptions,
    'generated_at':   time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}

with open(overall_path, 'w') as fh:
    json.dump(overall_artifact, fh, indent=2)

print(f'Saved combined artifact → {overall_path}')
print()
print('Description lengths by channel:')
for ch, data in channel_descriptions.items():
    desc_len = len(data.get('description', ''))
    print(f'  {ch}: {desc_len:,} chars')
